# Ranking Scorio GPQA models with `scorio.rank`

The ranking uses ten questions from one field, with 80 attempts per question. Scorio expects
an `L x M x N` tensor. The reader requests only correctness and identity columns from the
public Bucket.


In [1]:
from concurrent.futures import ThreadPoolExecutor

import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
from IPython.display import display

from scorio import rank

BUCKET_ROOT = "hf://buckets/harimo/scorio-gpqa"


def pool_path(model, question_id):
    return f"{BUCKET_ROOT}/data/{model}/super_gpqa/q{question_id:04d}.parquet"


def read_pools(model, question_ids, columns, max_workers=2):
    """Read selected columns from question files, preserving question order."""
    paths = [pool_path(model, q) for q in question_ids]

    def read_one(path):
        return pq.read_table(path, columns=columns)

    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        tables = list(executor.map(read_one, paths))
    return pa.concat_tables(tables).to_pandas()

models = ["Qwen3.6-35B-A3B", "gpt-oss-20b_low", "gpt-oss-20b_medium", "gpt-oss-20b_high"]
field_start = 0
question_count = 10
question_ids = range(field_start, field_start + question_count)
columns = ["full_data_id", "seed", "field", "evalscope_is_correct"]

matrices = []
field_name = None
for model in models:
    rows = read_pools(model, question_ids, columns).sort_values(["full_data_id", "seed"])
    assert rows.groupby("full_data_id").size().eq(80).all()
    field_name = rows.field.iloc[0]
    matrices.append(rows.evalscope_is_correct.to_numpy().astype(int).reshape(question_count, 80))

R = np.stack(matrices)
print("field:", field_name)
print(R.shape, "models x questions x seeds")


/tmp/scorio_uv_cache/archive-v0/qlDst6OSBXQ1PndVk4Wef/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


field: Aeronautical and Astronautical Science and Technology
(4, 10, 80) models x questions x seeds


## Bayes@N ranking


In [2]:
ranks, scores = rank.bayes(R, return_scores=True)
leaderboard = pd.DataFrame({"rank": ranks.astype(int), "Bayes@N": scores}, index=models)
display(leaderboard.sort_values("rank").round(3))


,rank,Bayes@N
Qwen3.6-35B-A3B,1,0.578
gpt-oss-20b_high,2,0.437
gpt-oss-20b_medium,3,0.374
gpt-oss-20b_low,4,0.233


## Compare ranking methods


In [3]:
methods = ["bayes", "borda", "win_rate", "bradley_terry", "elo", "rasch_mml", "thompson"]
comparison = pd.DataFrame(
    {method: getattr(rank, method)(R).astype(int) for method in methods},
    index=models,
).sort_values("bayes")
display(comparison)


,bayes,borda,win_rate,bradley_terry,elo,rasch_mml,thompson
Qwen3.6-35B-A3B,1,1,1,1,1,1,1
gpt-oss-20b_high,2,2,2,2,2,2,2
gpt-oss-20b_medium,3,3,3,3,3,3,3
gpt-oss-20b_low,4,4,4,4,4,4,4


## Ranking at different sample budgets


In [4]:
budgets = [1, 2, 4, 8, 16, 32, 80]
sweep = pd.DataFrame(
    {f"n={n}": rank.bayes(R[:, :, :n]).astype(int) for n in budgets},
    index=models,
).sort_values("n=80")
display(sweep)
print("models whose rank changes between n=1 and n=80:",
      int((sweep["n=1"] != sweep["n=80"]).sum()), "of", len(models))


,n=1,n=2,n=4,n=8,n=16,n=32,n=80
Qwen3.6-35B-A3B,1,1,1,1,1,1,1
gpt-oss-20b_high,2,2,3,2,2,2,2
gpt-oss-20b_medium,3,2,2,3,3,3,3
gpt-oss-20b_low,4,4,4,4,4,4,4


models whose rank changes between n=1 and n=80: 0 of 4


The [ranking reference](https://github.com/mohsenhariri/scorio/blob/main/scorio/rank/README.md)
describes the voting, paired-comparison, item-response, graph, and listwise methods.
